# 11 — Chuẩn bị khảo sát thực nghiệm trọng số trên BAF

    Mục đích: tạo protocol/cache/nguồn tác giả dùng chung. Bước này chưa huấn luyện cải tiến đề xuất. Đọc `docs/RESEARCH_METHODS.md` trước khi Run All. Các notebook12–15 lưu training thật trong notebook con, notebook16 mở test khi mọi suite đã chọn đã khóa.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = Path(
    os.environ.get("BAF_CACHE_ROOT") or ROOT / ".cache" / "baf-research"
).expanduser().resolve()
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Project: D:\UIT\DS317\BAF_project
Module: baf-research-20261028-v1


## 1. Chọn phạm vi trước khi tạo campaign

    Mặc định full BAF, ba seed, có LSR. LSR cần VAE + hai diffusion model và hàng trăm triệu lượt denoising; streaming giảm RAM/disk nhưng vẫn tốn thời gian. Nếu ngân sách chưa đủ, đặt `INCLUDE_LSR=False` trước khi chạy. Kết quả pilot chỉ kiểm kỹ thuật. Không sửa cấu hình khi resume campaign.

In [2]:
DATA_PATH = ROOT / "data" / "raw" / "Base.csv"
INCLUDE_LSR = True
RESUME_CAMPAIGN = None

CONFIG = gru.ResearchConfig(
    mode="full",
    seeds=(42, 317, 2026),
    threads=4,
    device="auto",
    meta_per_class=128,
    epochs=20,
    batch_size=2048,
    meta_batch_size=256,
    patience=5,
    tree_rounds=300,
    vae_epochs=30,
    diffusion_epochs=50,
    score_batch_size=2048,
    score_noise_chunk=4,
    bootstrap_repeats=1000,
)
DATASET_LABEL = "BAF_Base"
SUITES = ["losses", "meta", "xgboost"]
if INCLUDE_LSR:
    SUITES.insert(2, "lsr")

print("Selected suites:", SUITES)
print("Expected predictor fits:", (20 if INCLUDE_LSR else 14) * 5)
print("LSR auxiliary budgets:", CONFIG.vae_epochs, CONFIG.diffusion_epochs)


Selected suites: ['losses', 'meta', 'lsr', 'xgboost']
Expected predictor fits: 100
LSR auxiliary budgets: 30 50


## 2. Clone và kiểm nguồn; fit preprocessing chỉ trên core train

    Kiểm hash Base.csv, commit và file nguồn tác giả. Giữ meta cân bằng trong train tháng5; loại các hàng meta khỏi train của mọi learner. Validation tháng6, test tháng7. Chỉ preprocessing được áp dụng lên test ở đây; chưa chọn model hoặc tính test metric.

In [3]:
CAMPAIGN = gru.research_create_campaign(
    project=ROOT,
    data_path=DATA_PATH,
    config=CONFIG,
    resume=RESUME_CAMPAIGN,
    dataset_label=DATASET_LABEL,
    suites=SUITES,
)
print("Lưu đường dẫn này để dùng ở 12–16:", CAMPAIGN)


SOURCE_VERIFIED cb 1d7857208a2abc03d84e35a9d5383af8225d4b4d
SOURCE_VERIFIED ldam 2536330f2afdaa65618323cb5a5850efccce762a
SOURCE_VERIFIED lre 0b616c99ecf8a1c99925322167694272a966ed00
SOURCE_VERIFIED mwn 20c3276b9bf2a192dbaba17cdaef1fd280338d7f
SOURCE_VERIFIED lsr f8561071b72942e0001612bfea505aaeadbb4e73
SOURCE_VERIFIED imxgb 6d82dc83266a32c8579a3e8ea172378085ad1711
                 n fraud              months
train       794733  8023  [0, 1, 2, 3, 4, 5]
validation  108168  1450                 [6]
test         96843  1428                 [7]
meta           256   128                 [5]
RESEARCH_DATA_READY D:\UIT\DS317\BAF_project\results\research_data\80bcacc462d4de92
CAMPAIGN_READY D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e
Lưu đường dẫn này để dùng ở 12–16: D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e


## 3. Kiểm thông tin đã khóa

    Đây là manifest thực tế để đối chiếu số hàng, split và source. Sau cell này chạy12,13,14 (nếu chọn) và15 lần lượt. Giữ nguyên campaign; chưa chạy16 khi còn thiếu suite.

In [4]:
import pandas as pd

plan = gru.research_read(CAMPAIGN / "campaign.json")
data = gru.research_read(Path(plan["cache"]) / "metadata.json")
display(pd.DataFrame(data["counts"]).T)
print("Data SHA256:", data["data_sha256"])
print("Campaign file:", CAMPAIGN / "campaign.json")
print("Mode:", data["mode"])


,n,fraud,months
train,794733,8023,"[0, 1, 2, 3, 4, 5]"
validation,108168,1450,[6]
test,96843,1428,[7]
meta,256,128,[5]


Data SHA256: 7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809
Campaign file: D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e\campaign.json
Mode: full
